```python

from timex.ecg import ECGxtract

ecg_extractor = ECGxtract(data_location='/path/to/data', leads=['I', 'II',..], id_names=['ID'], iterative=True, sampling_rate=1000)

ecg_segments = ecg_extractor.extract_segments()
ecg_rates = ecg_extractor.extract_rates()
ecg_timings = ecg_extractor.extract_timings(ecg_segments)
ecg_cwt = ecg_extractor.extract_cwt(ecg_segments)
ecg_dwt = ecg_extractor.extract_dwt(ecg_segments)
ecg_ft = ecg_extractor.extract_ft(ecg_segments)
```

In [1]:
# add autoreload
%load_ext autoreload
%autoreload 2
import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb
import os
import sys
import re
import dotenv
from collections import defaultdict
from tqdm import tqdm

import matplotlib.pyplot as plt

In [2]:
dotenv.load_dotenv('../.env')

True

In [3]:
BASE_DIR = os.getenv('ECG_DIR')
HEA_DIR = os.path.join(BASE_DIR,
                       'physionet_classification_challenge',
                       'training',
                       'ptb', 
                       'g1')

SAMPLE_RATE = 500

In [4]:
hea_list = [os.path.join(HEA_DIR, f.split('.')[0]) 
            for f in os.listdir(HEA_DIR) if f.endswith('.hea')]

In [ ]:
ecg_path = '/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0513' # hea_list[5]
hea = os.path.basename(ecg_path)

band_names = wfdb.rdheader(ecg_path).__dict__['sig_name']
record = wfdb.rdrecord(ecg_path)
meta_str = ",".join(wfdb.rdheader(ecg_path).__dict__['comments'])

In [ ]:
record.__dict__['comments']

In [ ]:
re_age = re.compile(r'Age:\s?(\d+)') 
re_sex = re.compile(r'Sex:\s?(\w+)')
re_diagnosis = re.compile(r'(Dx|Diagnosis):\s?([0-9]+)', re.IGNORECASE)

age  = int(re.findall(re_age, meta_str)[0])
sex = re.findall(re_sex, meta_str)[0]
diagnosis = re.findall(re_diagnosis, meta_str)[0][1]

print(f"Age:{age},\nSex:{sex},\nDiagnosis:{diagnosis}")

print(ecg_path, meta_str)

In [ ]:
wfdb.rdrecord(ecg_path).fs

In [ ]:
tmp = wfdb.rdsamp(ecg_path)[0]

In [ ]:
tdf = pd.DataFrame(tmp, columns=band_names)
tdf['age'] = age
tdf['sex'] = sex
tdf['diagnosis'] = diagnosis
tdf['segment'] = hea

In [ ]:
band_names

In [ ]:
BAND = 'II'

In [ ]:
cleaned_signal_processed, r_peaks  = nk.ecg_process(tdf[BAND], sampling_rate=SAMPLE_RATE)
cleaned_signal = nk.ecg_clean(tdf[BAND], sampling_rate=SAMPLE_RATE)

In [ ]:
filtered_signal = nk.signal_filter(tdf[BAND], sampling_rate=SAMPLE_RATE, lowcut=0.5, highcut=40, order=2)
plt.plot(filtered_signal)
plt.plot(tdf[BAND])

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(tdf[BAND], label='Original Signal')
plt.plot(cleaned_signal, label='Cleaned Signal')
plt.title(f"ECG Signal - {BAND}")
plt.legend()

In [ ]:
_, peaks_info = nk.ecg_peaks(cleaned_signal, sampling_rate=SAMPLE_RATE)

peak_features_time = nk.hrv_time(peaks_info['ECG_R_Peaks'], sampling_rate=SAMPLE_RATE, show=False)

peak_features_nonlinear = nk.hrv_nonlinear(peaks_info['ECG_R_Peaks'], 
                                           sampling_rate=SAMPLE_RATE, 
                                           show=False)

peak_features_frequency = nk.hrv_frequency(peaks_info['ECG_R_Peaks'], sampling_rate=SAMPLE_RATE, show=False, 
                                           normalize=True)
                                        

In [ ]:
nk.hrv_frequency(peaks_info['ECG_R_Peaks'], sampling_rate=SAMPLE_RATE, show=False, normalize=True).to_dict(orient='index')[0]                                        

In [ ]:
nk.hrv_nonlinear(peaks_info['ECG_R_Peaks'],sampling_rate=SAMPLE_RATE,show=False).to_dict(orient='index')[0]


In [ ]:
nk.hrv_time(peaks_info['ECG_R_Peaks'], sampling_rate=SAMPLE_RATE).to_dict(orient='index')[0]

In [ ]:
signals, _ = nk.ecg_process(cleaned_signal, sampling_rate=1000)
hr_features = nk.ecg_intervalrelated(signals)
hr_features = {k: np.array(v).flatten()[0] for k,v in hr_features.to_dict(orient='index')[0].items()}
hr_features

In [ ]:
def band_power(freqs, power, low, high):
    indices = np.logical_and(freqs >= low, freqs <= high)
    return np.trapz(power[indices], freqs[indices])

def spectral_edge_freq(freqs, power, percent=95):
    total_power = np.trapz(power, freqs)
    target_power = total_power * percent/100
    cumulative_power = np.cumsum(power) * (freqs[1] - freqs[0])
    idx = np.where(cumulative_power >= target_power)[0][0]
    return freqs[idx]

def spectral_entropy(power):
    power = power[power > 0]  # Remove zero values
    power_normalized = power / np.sum(power)
    entropy = -np.sum(power_normalized * np.log2(power_normalized+1e-10))
    return entropy

def spectral_flatness(power):
    geometric_mean = np.exp(np.mean(np.log(power + 1e-10)))
    arithmetic_mean = np.mean(power)
    return geometric_mean / arithmetic_mean


psd_chars = nk.signal_psd(cleaned_signal, sampling_rate=SAMPLE_RATE, method='welch').values
FREQS = psd_chars[:,0]
POWER = psd_chars[:,1]

# frequency of maximum power
dominant_frequency = FREQS[np.argmax(POWER)]

# Median frequency, frequency that divides the cumulative power spectrum in two equal parts 
total_power = np.trapz(POWER, FREQS)
cumulative_power = np.cumsum(POWER) * (FREQS[1] - FREQS[0])
median_freq_idx = np.where(cumulative_power >= POWER/2)[0][0]
median_freq = FREQS[median_freq_idx]

######
p_wave_band = (0.5, 3)      # P-wave components
qrs_complex_band = (4, 20)   # QRS complex components
t_wave_band = (0.5, 7)      # T-wave components
baseline_band = (0, 0.5)     # Baseline wander
mains_noise_band = (49, 51)  # 50Hz power line interference
low_freq_band = (0.04, 0.15) # low frequency
high_freq_band = (0.15, 0.4) # high frequency

p_wave_band_power = band_power(FREQS, POWER, *p_wave_band) # delta
qrs_complex_band_power = band_power(FREQS, POWER, *qrs_complex_band) # theta
t_wave_band_power = band_power(FREQS, POWER, *t_wave_band) # alpha
baseline_band_power = band_power(FREQS, POWER, *baseline_band) # beta
mains_noise_band_power = band_power(FREQS, POWER, *mains_noise_band) # gamma
noise_band_power = band_power(FREQS, POWER, 40, 100) # noise
lf_power = band_power(FREQS, POWER, *low_freq_band) # low frequency
hf_power = band_power(FREQS, POWER, *high_freq_band) # high frequency
total_power = np.trapz(POWER, FREQS) # total power

# ratio of low frequency to high frequency power
lf_hf_ratio = lf_power / hf_power
# ratio of low frequency to total power
lf_total_ratio = lf_power / total_power
qrs_noise_ratio = qrs_complex_band_power / noise_band_power

#######
_spectral_edge_freq = spectral_edge_freq(FREQS, POWER)

#######
_spectral_entropy = spectral_entropy(POWER)

#######
_spectral_flatness = spectral_flatness(POWER)

#######
spectral_centroid = np.sum(FREQS * POWER) / np.sum(POWER)

In [ ]:
_spectral_edge_freq, _spectral_entropy, _spectral_flatness, spectral_centroid

In [ ]:
_, waves_peak = nk.ecg_delineate(cleaned_signal, 
                                 peaks_info, 
                                 sampling_rate=SAMPLE_RATE, 
                                 method="dwt",
                                 show=True,
                                 show_type='all')

In [ ]:
# Unacceptable, Barely acceptable, Excellent
quality_string = nk.ecg_quality(cleaned_signal, 
                                rpeaks=peaks, 
                                method='zhao2018',
                                approach='fuzzy',
                                sampling_rate=SAMPLE_RATE)

# Unacceptable, Barely acceptable, Excellent
quality_score = nk.ecg_quality(cleaned_signal, 
                               sampling_rate=SAMPLE_RATE)

ecg_rates = nk.ecg_rate(peaks, sampling_rate=SAMPLE_RATE, desired_length=len(peaks)+1)

ecg_rsp = nk.ecg_rsp(ecg_rates, method='vangent2019', sampling_rate=SAMPLE_RATE)

In [ ]:
ecg_rates.shape, ecg_rsp.shape, cleaned_signal.shape

In [ ]:
ecg_segmentation = nk.ecg_segment(cleaned_signal,
                                  sampling_rate=SAMPLE_RATE,
                                  show=True)

In [5]:
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
import extractor
import wavelets
import ecg

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

In [7]:
ecgDS = ecg.ECGDataset(df=os.path.join(BASE_DIR,'physionet_classification_challenge', 'training'),
                       label_binarizer=None,
                       augmentations=[],
                       preprocessing=[])

In [ ]:
res_dict = defaultdict(dict)
for i in tqdm(range(len(ecgDS))):
    r = ecgDS[i]
    res_dict[r[0]]={
        'signal': r[2],
        'label': r[3]
    }
    res_dict[r[0]].update(r[4])

 12%|█▏        | 4972/43101 [00:32<03:05, 205.70it/s]

In [ ]:
_keys = list(res_dict.keys())

In [ ]:
ecgExtractor = ecg.ECGxtract(sampling_rate=1000, 
                             sanity_check=True,
                             extractor_groups=['wavelets'], 
                             extractor_type='catch22')

In [ ]:
extractions = ecgExtractor.extract_from_dict(res_dict, SignalCol='signal', FsCol='fs')

100%|██████████| 12/12 [00:07<00:00,  1.56it/s]


In [ ]:
extractions_df = pd.DataFrame(extractions).T

In [ ]:
# add label and demographics
extractions_df

,CHANNEL_0_dominant_frequency,CHANNEL_0_median_freq,CHANNEL_0_p_wave_band_power,CHANNEL_0_qrs_complex_band_power,CHANNEL_0_t_wave_band_power,CHANNEL_0_baseline_band_power,CHANNEL_0_main_noise_band_power,CHANNEL_0_noise_band_power,CHANNEL_0_lf_power,CHANNEL_0_hf_power,...,CHANNEL_11_Frequency 7,CHANNEL_11_Frequency 8,CHANNEL_11_Phase 1,CHANNEL_11_Phase 2,CHANNEL_11_Phase 3,CHANNEL_11_Phase 4,CHANNEL_11_Phase 5,CHANNEL_11_Phase 6,CHANNEL_11_Phase 7,CHANNEL_11_Phase 8
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0513.hea,1.116555,8.890778,0.052676,0.317663,0.170758,1.135911e-04,0.000050,0.000573,6.886310e-10,2.022995e-05,...,0.000975,0.000872,1.925552,1.991108,1.890040,1.869325,2.141651,2.038375,2.101099,2.260388
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0172.hea,1.154534,5.711905,0.070404,0.286951,0.266064,2.636674e-05,0.000001,0.000017,4.558778e-10,3.916329e-06,...,0.000962,0.000802,2.205543,1.751072,-0.309740,-0.151419,0.148689,0.811487,0.209349,-0.077638
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0377.hea,2.838542,5.651042,0.201565,0.428375,0.363809,1.475587e-03,0.000018,0.000236,1.489222e-08,2.415659e-04,...,0.002886,0.003848,-2.951319,-2.138676,2.892281,-1.006964,0.276887,0.146401,-1.397234,2.597614
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0310.hea,2.682338,5.355996,0.052589,0.094047,0.101345,9.710494e-07,0.000005,0.000173,5.391440e-11,4.073216e-07,...,0.000962,0.000909,3.084323,3.136153,-3.133258,3.129617,3.137758,3.129054,-3.140884,-3.139078
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0188.hea,2.343791,8.177225,0.107328,0.519039,0.295566,1.320952e-04,0.000001,0.000029,2.963622e-09,3.496922e-05,...,0.000802,0.000909,0.523341,0.454760,0.636937,-0.965717,0.039934,0.216915,0.074300,0.148460
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0482.hea,0.958763,5.391753,0.095923,0.213461,0.203505,4.715225e-04,0.000005,0.000046,1.539117e-09,4.416221e-05,...,0.001079,0.001143,-1.102199,-0.534132,-1.357321,-1.082817,-1.181720,-0.562930,-1.274570,-1.304902
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0470.hea,0.966570,2.933040,0.185543,0.130073,0.250485,6.593181e-05,0.000004,0.000076,4.133702e-10,5.259357e-06,...,0.000770,0.000821,0.139779,-0.001153,0.095623,0.062924,-0.023081,0.027822,0.017159,0.023751
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0375.hea,1.041667,4.140625,0.145359,0.177851,0.224137,6.679396e-06,0.000003,0.000041,8.231677e-09,9.094868e-07,...,0.003047,0.002566,-0.066539,-0.079575,-0.164864,-0.961266,-0.563285,-0.209400,-0.463343,-0.724011
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0313.hea,2.638935,5.355996,0.092561,0.256422,0.198930,2.393306e-05,0.000005,0.000080,4.135059e-09,3.294271e-06,...,0.000374,0.000428,-3.141072,-3.140665,-3.140489,-3.140659,-3.141298,3.140656,3.138712,3.135915
/media/bramiozo/DATA-FAST/ecg/physionet_classification_challenge/training/ptb/g1/S0106.hea,3.003524,9.158145,0.057938,0.611487,0.241938,5.358899e-05,0.000025,0.000671,2.408161e-10,1.468366e-05,...,0.000962,0.000909,-2.989881,-3.069658,-3.020370,-3.033544,-3.063235,-3.038272,-3.065751,-3.074689
